# 04b - Train resnet18_imagenet_covidqu

This notebook runs one ResNet18 + SimCLR experiment. It only prepares Colab and calls repository scripts with `!python`.


## 1. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Clone Or Pull Repo


In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/tlinhevg05/contrastive-synthesis-medcls_CVProject.git'
REPO_ROOT = Path('/content/contrastive-synthesis-medcls_CVProject')

if REPO_ROOT.exists():
    %cd {REPO_ROOT}
    !git pull
else:
    %cd /content
    !git clone {REPO_URL}
    %cd {REPO_ROOT}

print('Repo:', Path.cwd())


## 3. Install Minimal Dependencies

Colab already provides PyTorch and torchvision.


In [ ]:
!pip install -q scikit-learn matplotlib pandas Pillow PyYAML

import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 4. Editable Runner Variables

Use `PRETRAIN_EPOCH_OVERRIDE` to control SimCLR cost. Keep `FINETUNE_EPOCH_OVERRIDE = None` to use the config fine-tuning budget of 70 epochs for fair comparison with ResNet baselines.


In [ ]:
REPO_ROOT = Path('/content/contrastive-synthesis-medcls_CVProject')
OUTPUT_ROOT = Path('/content/drive/MyDrive/medcls_cvproject/results/experiments')
SYNTHETIC_MANIFEST = Path('/content/drive/MyDrive/medcls_cvproject/data/manifests/synthetic_dcgan.csv')

RUN_PRETRAIN = True
RUN_FINETUNE = False
PRETRAIN_EPOCH_OVERRIDE = 10  # Total target pretraining epochs. Increase to 20, 30, ... to resume in chunks.
FINETUNE_EPOCH_OVERRIDE = None  # Keep None for config 70 and fair baseline comparison.

pretrain_epoch_arg = '' if PRETRAIN_EPOCH_OVERRIDE is None else f'--epochs {PRETRAIN_EPOCH_OVERRIDE}'
finetune_epoch_arg = '' if FINETUNE_EPOCH_OVERRIDE is None else f'--epochs {FINETUNE_EPOCH_OVERRIDE}'

%cd {REPO_ROOT}
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('SYNTHETIC_MANIFEST:', SYNTHETIC_MANIFEST)
print('RUN_PRETRAIN:', RUN_PRETRAIN)
print('RUN_FINETUNE:', RUN_FINETUNE)
print('pretrain_epoch_arg:', pretrain_epoch_arg)
print('finetune_epoch_arg:', finetune_epoch_arg)


## 5. Lightweight Checks


In [ ]:
!python scripts/check_experiment_inputs.py --synthetic-manifest "{SYNTHETIC_MANIFEST}"
!python -m py_compile scripts/run_simclr_resnet.py scripts/run_classification_resnet.py


## 6. Experiment: resnet18_imagenet_covidqu

SimCLR pretraining from ImageNet initialization on real unlabeled COVID-QU, then supervised fine-tuning on real labeled manifests.

Run `Pretrain Only` repeatedly by increasing `PRETRAIN_EPOCH_OVERRIDE`. Run `Fine-Tune Only` only after pretraining reaches the epoch target you want to report.


### 6a. Pretrain Only: resnet18_imagenet_covidqu


In [ ]:
EXP = 'resnet18_imagenet_covidqu'
OUT = OUTPUT_ROOT / EXP
CKPT = OUT / 'pretrain/checkpoints/best_simclr_backbone.pth'
RESUME_CKPT = OUT / 'pretrain/checkpoints/last_simclr_checkpoint.pth'

print('OUT:', OUT)
print('CKPT:', CKPT, 'exists=', CKPT.exists())
print('RESUME_CKPT:', RESUME_CKPT, 'exists=', RESUME_CKPT.exists())

if RUN_PRETRAIN:
    !python scripts/run_simclr_resnet.py \
      --config configs/experiments/resnet18/imagenet_covidqu.yaml \
      --output-dir "{OUT}" \
      --resume-checkpoint "{RESUME_CKPT}" \
      {pretrain_epoch_arg}
else:
    print('Skipping pretrain', EXP)


### 6b. Fine-Tune Only: resnet18_imagenet_covidqu


In [ ]:
EXP = 'resnet18_imagenet_covidqu'
OUT = OUTPUT_ROOT / EXP
CKPT = OUT / 'pretrain/checkpoints/best_simclr_backbone.pth'

print('CKPT:', CKPT, 'exists=', CKPT.exists())

if RUN_FINETUNE:
    if not CKPT.exists():
        raise FileNotFoundError(f'SimCLR checkpoint not found: {CKPT}. Finish pretraining first.')
    !python scripts/run_classification_resnet.py \
      --config configs/experiments/resnet18/imagenet_covidqu.yaml \
      --manifest-dir data/manifests \
      --output-dir "{OUT}" \
      --pretrained-checkpoint "{CKPT}" \
      {finetune_epoch_arg}
else:
    print('Skipping finetune', EXP)


## 7. Display Result


In [ ]:
import json
import pandas as pd

metrics_path = OUT / 'metrics.json'
if metrics_path.exists():
    display(pd.DataFrame([{**{'experiment_id': EXP}, **json.loads(metrics_path.read_text())}]))
else:
    print('No metrics yet:', metrics_path)

!find "{OUT}" -maxdepth 3 -type f | sort || true
